# LeetCode 2042 — Check if Numbers Are Ascending in a Sentence bằng Python

Link đề: https://leetcode.com/problems/check-if-numbers-are-ascending-in-a-sentence/ · Độ khó: **Easy** · Chủ đề: String

```python
class Solution:
    def areNumbersAscending(self, s: str) -> bool:

```

**Mục tiêu học**

1. Biết tách câu thành **token** bằng `str.split()` và nhận diện token là số bằng `str.isdigit()`.
2. Phân biệt **so sánh chuỗi** (`"12" < "6"` là `True`!) với **so sánh số** (`int`).
3. Nắm kỹ thuật "**giữ phần tử trước** (`prev`)" để kiểm tra dãy **tăng nghiêm ngặt** trong một lần duyệt.
4. Thêm cách duyệt từng ký tự tự ghép số (không dùng `split`) và cách dùng `re.findall`, `zip`, `all`.
5. Nộp `class Solution`, test kỹ các trường hợp số bằng nhau, số nhiều chữ số.

## 1. Đề bài yêu cầu gì?

Một **câu** là các token cách nhau **đúng một dấu cách**, không có dấu cách ở đầu/cuối. Mỗi token hoặc là **số dương** (chữ số 0–9, không có số 0 đứng đầu), hoặc là **từ** gồm chữ thường.

Kiểm tra: **mọi số trong câu có tăng nghiêm ngặt từ trái sang phải không** (mỗi số < số đứng sau nó).

| Ví dụ | s | Các số | Output |
|---|---|---|---|
| 1 | `"1 box has 3 blue 4 red 6 green and 12 yellow marbles"` | 1, 3, 4, 6, 12 | `True` |
| 2 | `"hello world 5 x 5"` | 5, 5 | `False` (bằng nhau không phải tăng nghiêm ngặt) |
| 3 | `"sunset is at 7 51 pm overnight lows will be in the low 50 and 60 s"` | 7, 51, 50, 60 | `False` (51 > 50) |

Constraints chính: `3 <= len(s) <= 200`; có từ 2 đến 100 token; **có ít nhất 2 số**; mỗi số `< 100`, dương, không có số 0 đầu.

In [1]:
s = "1 box has 3 blue 4 red 6 green and 12 yellow marbles"
print(s.split())

['1', 'box', 'has', '3', 'blue', '4', 'red', '6', 'green', 'and', '12', 'yellow', 'marbles']


## 2. Nền tảng Python

### 2.1. `str.split()` — tách chuỗi thành list token

- `s.split()` (không tham số): tách theo **mọi khoảng trắng liên tiếp**, bỏ khoảng trắng đầu/cuối.
- `s.split(" ")`: tách theo **đúng một dấu cách**; hai dấu cách liền nhau tạo token rỗng `""`.

Bài này đảm bảo đúng một dấu cách nên cả hai đều được, nhưng `split()` an toàn hơn.

In [2]:
print("a  b c".split())      # ['a', 'b', 'c']
print("a  b c".split(" "))   # ['a', '', 'b', 'c']  <- token rỗng
print(" x y ".split())       # ['x', 'y']

['a', 'b', 'c']
['a', '', 'b', 'c']
['x', 'y']


### 2.2. `str.isdigit()` — token có phải số không?

`"12".isdigit()` → `True` khi **mọi** ký tự là chữ số và chuỗi khác rỗng. Theo đề, token hoặc toàn chữ số, hoặc toàn chữ cái, nên chỉ cần kiểm tra **ký tự đầu**: `tok[0].isdigit()` (đúng như gợi ý của đề).

In [3]:
for tok in ["12", "box", "7", "a1", ""]:
    print(repr(tok), "->", tok.isdigit())
print("ký tự đầu của '51':", "51"[0], "51"[0].isdigit())

'12' -> True
'box' -> False
'7' -> True
'a1' -> False
'' -> False
ký tự đầu của '51': 5 True


### 2.3. Chuỗi so sánh khác số — cái bẫy lớn nhất

So sánh chuỗi là **so sánh từ điển** (lexicographic): so ký tự đầu trước. `"12" < "6"` vì `'1' < '6'`. Phải đổi sang số bằng `int(tok)` trước khi so sánh.

In [4]:
print("12" < "6", 12 < 6)        # True False  <- chuỗi so sánh SAI ý đồ
print("9" < "10", 9 < 10)        # False True
print(int("51"), type(int("51")))

True False
False True
51 <class 'int'>


### 2.4. Kiểm tra dãy tăng nghiêm ngặt bằng `prev`

Ý tưởng: duyệt từ trái sang phải, nhớ số trước đó `prev`. Nếu gặp số `x` mà `x <= prev` → `False` ngay. Khởi tạo `prev = 0` an toàn vì mọi số trong đề **dương** (≥ 1).

In [5]:
nums = [1, 3, 4, 6, 12]
prev = 0
for x in nums:
    print(f"prev={prev}, x={x}, x > prev? {x > prev}")
    prev = x

prev=0, x=1, x > prev? True
prev=1, x=3, x > prev? True
prev=3, x=4, x > prev? True
prev=4, x=6, x > prev? True
prev=6, x=12, x > prev? True


### 2.5. `zip(a, a[1:])`, `all(...)` và slicing

- `a[1:]` là list từ phần tử thứ 1 tới cuối (slicing `start:end`).
- `zip(a, a[1:])` tạo các **cặp liền kề** `(a[0],a[1]), (a[1],a[2]), ...`.
- `all(điều_kiện for ...)` trả `True` nếu **mọi** phần tử thỏa (và dừng sớm khi gặp phần tử sai).

In [6]:
a = [7, 51, 50, 60]
print(a[1:])
print(list(zip(a, a[1:])))
print(all(x < y for x, y in zip(a, a[1:])))    # False vì 51 < 50 sai
print(all([]))                                  # True: không có cặp nào vi phạm

[51, 50, 60]
[(7, 51), (51, 50), (50, 60)]
False
True


## 3. Cách 1 — Trực giác: lấy hết các số ra rồi kiểm tra

Hai bước tách bạch, dễ đọc:

1. `nums = [int(t) for t in s.split() if t[0].isdigit()]`
2. Kiểm tra `nums` tăng nghiêm ngặt.

In [7]:
def ascending_collect(s):
    nums = [int(t) for t in s.split() if t[0].isdigit()]
    for i in range(1, len(nums)):
        if nums[i] <= nums[i - 1]:
            return False
    return True

print(ascending_collect("1 box has 3 blue 4 red 6 green and 12 yellow marbles"))   # True
print(ascending_collect("hello world 5 x 5"))                                       # False
print(ascending_collect("sunset is at 7 51 pm overnight lows will be in the low 50 and 60 s"))  # False

True
False
False


### Có gì chưa tối ưu?

Cách 1 đúng và đã `O(n)`. Điểm có thể cải thiện:

- Tạo thêm list `nums` (bộ nhớ `O(số lượng số)`).
- Luôn tách toàn bộ câu, kể cả khi vi phạm đã xuất hiện ngay đầu câu.

**Insight:** để kiểm tra "tăng", ta chỉ cần **so số hiện tại với số liền trước** — không cần nhớ cả danh sách. Gộp hai bước thành một vòng lặp với biến `prev`, dừng ngay khi vi phạm.

## 4. Cách 2 (lời giải chính) — Một vòng lặp với `prev`

Từng mảnh:

- `for tok in s.split():` duyệt token.
- `if tok[0].isdigit():` chỉ xử lý số; từ thì bỏ qua.
- `x = int(tok)`; nếu `x <= prev` → `return False`; ngược lại `prev = x`.
- Hết vòng → `return True`.

In [8]:
def ascending_prev(s):
    prev = 0                         # mọi số trong đề >= 1 nên 0 là "nhỏ hơn tất cả"
    for tok in s.split():
        if tok[0].isdigit():
            x = int(tok)
            if x <= prev:            # bằng nhau cũng là vi phạm (tăng NGHIÊM NGẶT)
                return False
            prev = x
    return True

for s in ["1 box has 3 blue 4 red 6 green and 12 yellow marbles",
          "hello world 5 x 5",
          "sunset is at 7 51 pm overnight lows will be in the low 50 and 60 s"]:
    print(ascending_prev(s))

True
False
False


## 5. TRACE từng bước

Ví dụ 3: `"sunset is at 7 51 pm overnight lows will be in the low 50 and 60 s"`.

| token | là số? | x | prev trước | x > prev? | hành động |
|---|---|---|---|---|---|
| sunset, is, at | không | – | 0 | – | bỏ qua |
| 7 | có | 7 | 0 | đúng | prev = 7 |
| 51 | có | 51 | 7 | đúng | prev = 51 |
| pm … low | không | – | 51 | – | bỏ qua |
| 50 | có | 50 | 51 | **sai** | `return False` |

Token `60` và `s` không bao giờ được xét — dừng sớm.

In [9]:
def trace_ascending(s):
    prev = 0
    for i, tok in enumerate(s.split()):
        if not tok[0].isdigit():
            print(f"[{i:2}] '{tok}': là từ, bỏ qua")
            continue
        x = int(tok)
        if x <= prev:
            print(f"[{i:2}] '{tok}': x={x} <= prev={prev} -> return False")
            return False
        print(f"[{i:2}] '{tok}': x={x} > prev={prev} -> prev = {x}")
        prev = x
    print("Duyệt hết, không vi phạm -> return True")
    return True

trace_ascending("sunset is at 7 51 pm overnight lows will be in the low 50 and 60 s")

[ 0] 'sunset': là từ, bỏ qua
[ 1] 'is': là từ, bỏ qua
[ 2] 'at': là từ, bỏ qua
[ 3] '7': x=7 > prev=0 -> prev = 7
[ 4] '51': x=51 > prev=7 -> prev = 51
[ 5] 'pm': là từ, bỏ qua
[ 6] 'overnight': là từ, bỏ qua
[ 7] 'lows': là từ, bỏ qua
[ 8] 'will': là từ, bỏ qua
[ 9] 'be': là từ, bỏ qua
[10] 'in': là từ, bỏ qua
[11] 'the': là từ, bỏ qua
[12] 'low': là từ, bỏ qua
[13] '50': x=50 <= prev=51 -> return False


False

`continue` nghĩa là "bỏ phần còn lại của vòng lặp này, sang token tiếp theo". Trace ví dụ 1 để thấy trường hợp trả `True` và số 2 chữ số `12`:

In [10]:
trace_ascending("1 box has 3 blue 4 red 6 green and 12 yellow marbles")

[ 0] '1': x=1 > prev=0 -> prev = 1
[ 1] 'box': là từ, bỏ qua
[ 2] 'has': là từ, bỏ qua
[ 3] '3': x=3 > prev=1 -> prev = 3
[ 4] 'blue': là từ, bỏ qua
[ 5] '4': x=4 > prev=3 -> prev = 4
[ 6] 'red': là từ, bỏ qua
[ 7] '6': x=6 > prev=4 -> prev = 6
[ 8] 'green': là từ, bỏ qua
[ 9] 'and': là từ, bỏ qua
[10] '12': x=12 > prev=6 -> prev = 12
[11] 'yellow': là từ, bỏ qua
[12] 'marbles': là từ, bỏ qua
Duyệt hết, không vi phạm -> return True


True

## 6. Cách 3 — Duyệt từng ký tự, tự ghép số (không dùng `split`)

Đây là cách "thủ công", hữu ích khi ngôn ngữ không có `split` hoặc khi muốn hiểu cách số được đọc từ chuỗi:

- Gặp chữ số `ch`: `cur = cur * 10 + (ord(ch) - ord('0'))` — giống công thức đảo số nhưng **ghép xuôi**.
- Gặp ký tự không phải chữ số (hoặc hết chuỗi) mà `cur > 0` → một số vừa kết thúc → so với `prev`.

`ord(ch) - ord('0')` đổi ký tự `'7'` thành số `7` (mã ASCII của các chữ số liên tiếp nhau).

In [11]:
print(ord('0'), ord('7'), ord('7') - ord('0'))
cur = 0
for ch in "51":
    cur = cur * 10 + (ord(ch) - ord('0'))
    print("đọc", ch, "-> cur =", cur)

48 55 7
đọc 5 -> cur = 5
đọc 1 -> cur = 51


In [12]:
def ascending_charscan(s):
    prev = 0
    cur = 0
    for ch in s + " ":               # thêm dấu cách "lính canh" để số cuối câu cũng được chốt
        if ch.isdigit():
            cur = cur * 10 + (ord(ch) - ord('0'))
        else:
            if cur > 0:              # vừa kết thúc một số (số dương nên cur > 0)
                if cur <= prev:
                    return False
                prev = cur
                cur = 0
    return True

print(ascending_charscan("1 box has 3 blue 4 red 6 green and 12 yellow marbles"))
print(ascending_charscan("hello world 5 x 5"))
print(ascending_charscan("a 1 b 2 c 30"))      # số ở cuối câu

True
False
True


Mẹo "**lính canh**" (sentinel) `s + " "`: nếu câu kết thúc bằng số, không có ký tự nào sau nó để "chốt" số. Thêm một dấu cách vào cuối giúp không phải viết thêm đoạn xử lý sau vòng lặp.

## 7. Cách 4 — Biểu thức chính quy `re.findall` + `all`

Module chuẩn `re` tìm mẫu trong chuỗi. `r"\d+"` nghĩa là "một hoặc nhiều chữ số liên tiếp". `re.findall` trả list mọi đoạn khớp.

In [13]:
import re
print(re.findall(r"\d+", "sunset is at 7 51 pm low 50 and 60 s"))

def ascending_regex(s):
    nums = list(map(int, re.findall(r"\d+", s)))
    return all(a < b for a, b in zip(nums, nums[1:]))

print(ascending_regex("1 box has 3 blue 4 red 6 green and 12 yellow marbles"))
print(ascending_regex("hello world 5 x 5"))

['7', '51', '50', '60']
True
False


`map(int, lst)` áp `int` lên từng phần tử; bọc `list(...)` để có list thật (vì ta cần slicing `nums[1:]`).

## 8. Code nộp LeetCode

In [14]:
class Solution:
    def areNumbersAscending(self, s: str) -> bool:
        prev = 0
        for tok in s.split():
            if tok[0].isdigit():
                x = int(tok)
                if x <= prev:
                    return False
                prev = x
        return True

sol = Solution()
print(sol.areNumbersAscending("1 box has 3 blue 4 red 6 green and 12 yellow marbles"))
print(sol.areNumbersAscending("hello world 5 x 5"))

True
False


## 9. Vì sao đúng?

**Định nghĩa:** dãy `a1, a2, ..., am` tăng nghiêm ngặt ⇔ với mọi `i`, `a_i < a_{i+1}` (chỉ cần các **cặp liền kề**, vì `<` có tính bắc cầu).

**Invariant:** trước khi xét số thứ `j`, `prev` là số thứ `j-1` (hoặc 0 nếu `j` là số đầu) và mọi cặp liền kề trong `j-1` số đầu đều thỏa `<`.

- Nếu `x <= prev` → cặp liền kề `(prev, x)` vi phạm → trả `False` là đúng.
- Nếu `x > prev` → cặp mới thỏa, cập nhật `prev = x`, invariant giữ.
- Duyệt hết không vi phạm → mọi cặp liền kề thỏa → `True`.

Khởi tạo `prev = 0` đúng vì số đầu tiên luôn ≥ 1 > 0, nên lần so đầu tiên không bao giờ trả sai `False`.

## 10. Độ phức tạp

Gọi `n = len(s)` (≤ 200).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Gom list số | `O(n)` | `O(n)` | list token + list số |
| 2. `split` + `prev` | `O(n)` | `O(n)` cho list token của `split` | dừng sớm khi vi phạm |
| 3. Quét ký tự | `O(n)` | `O(1)` thêm (chuỗi `s + " "` tạo bản sao `O(n)`; có thể tránh bằng xử lý sau vòng lặp) | không tạo token |
| 4. Regex + all | `O(n)` | `O(n)` | ngắn, cần biết regex |

## 11. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<34} PASS ({actual})")

tests = [
    ("ví dụ 1", "1 box has 3 blue 4 red 6 green and 12 yellow marbles", True),
    ("ví dụ 2 (bằng nhau)", "hello world 5 x 5", False),
    ("ví dụ 3 (giảm giữa câu)", "sunset is at 7 51 pm overnight lows will be in the low 50 and 60 s", False),
    ("chỉ 2 số, tăng", "1 2", True),
    ("chỉ 2 số, giảm", "2 1", False),
    ("bẫy so sánh chuỗi 9 < 10", "a 9 b 10", True),
    ("bẫy so sánh chuỗi 12 vs 6", "x 12 y 6", False),
    ("số cuối câu", "a 1 b 2 c 99", True),
    ("số liền nhau", "1 2 3 4 5 6 7 8 9 10", True),
]
fs = [ascending_collect, ascending_prev, ascending_charscan, ascending_regex,
      Solution().areNumbersAscending]
for name, s, exp in tests:
    for f in fs:
        assert f(s) == exp, (f.__name__, s)
    check(name, sol.areNumbersAscending(s), exp)

ví dụ 1                            PASS (True)
ví dụ 2 (bằng nhau)                PASS (False)
ví dụ 3 (giảm giữa câu)            PASS (False)
chỉ 2 số, tăng                     PASS (True)
chỉ 2 số, giảm                     PASS (False)
bẫy so sánh chuỗi 9 < 10           PASS (True)
bẫy so sánh chuỗi 12 vs 6          PASS (False)
số cuối câu                        PASS (True)
số liền nhau                       PASS (True)


In [16]:
import random
random.seed(2042)
words = ["a", "box", "cat", "x", "red"]
for _ in range(1000):
    m = random.randint(2, 8)
    nums = [random.randint(1, 99) for _ in range(m)]
    toks = []
    for x in nums:
        toks += random.sample(words, random.randint(0, 2)) + [str(x)]
    s = " ".join(toks)
    expected = all(a < b for a, b in zip(nums, nums[1:]))
    assert all(f(s) == expected for f in fs), s
print("1000 câu ngẫu nhiên: 5 cài đặt khớp nhau — PASS")

1000 câu ngẫu nhiên: 5 cài đặt khớp nhau — PASS


`" ".join(toks)` ghép list token thành chuỗi, ngăn cách bằng một dấu cách — ngược lại với `split`. `random.sample(words, k)` lấy `k` phần tử khác nhau ngẫu nhiên.

## 12. Lỗi người mới hay gặp

### Lỗi 1: So sánh chuỗi thay vì số

In [17]:
def wrong_string_compare(s):
    prev = ""
    for tok in s.split():
        if tok[0].isdigit():
            if tok <= prev:          # so sánh CHUỖI
                return False
            prev = tok
    return True

print("Sai :", wrong_string_compare("a 9 b 10"))     # False vì "10" < "9" theo từ điển
print("Đúng:", ascending_prev("a 9 b 10"))           # True

Sai : False
Đúng: True


**Sửa:** `int(tok)`.

### Lỗi 2: Dùng `<` thay vì `<=` khi phát hiện vi phạm

`if x < prev: return False` để lọt trường hợp **bằng nhau** (ví dụ 2 `"5 x 5"` → trả `True` sai).

In [18]:
def wrong_non_strict(s):
    prev = 0
    for tok in s.split():
        if tok[0].isdigit():
            x = int(tok)
            if x < prev:
                return False
            prev = x
    return True

print("Sai :", wrong_non_strict("hello world 5 x 5"))
print("Đúng:", ascending_prev("hello world 5 x 5"))

Sai : True
Đúng: False


### Lỗi 3: Đọc từng ký tự số như một số riêng

`for ch in s: if ch.isdigit(): x = int(ch)` biến `"12"` thành hai số `1` rồi `2` → ví dụ 1 sẽ thấy `... 6, 1, 2` và trả `False` sai.

In [19]:
def wrong_per_char(s):
    prev = 0
    for ch in s:
        if ch.isdigit():
            x = int(ch)
            if x <= prev:
                return False
            prev = x
    return True

print("Sai :", wrong_per_char("1 box has 3 blue 4 red 6 green and 12 yellow marbles"))
print("Đúng:", ascending_prev("1 box has 3 blue 4 red 6 green and 12 yellow marbles"))

Sai : False
Đúng: True


### Lỗi 4: Quên cập nhật `prev`

Không viết `prev = x` thì luôn so với 0 → hầu như luôn `True`.

### Lỗi 5: Quét ký tự mà quên chốt số cuối câu

Ở cách 3, nếu bỏ lính canh `s + " "` và không xử lý sau vòng lặp, số cuối câu (vd `"a 5 b 3"` → số 3) không được so sánh.

In [20]:
def wrong_no_sentinel(s):
    prev = cur = 0
    for ch in s:
        if ch.isdigit():
            cur = cur * 10 + int(ch)
        elif cur > 0:
            if cur <= prev:
                return False
            prev, cur = cur, 0
    return True

print("Sai :", wrong_no_sentinel("a 5 b 3"))     # True (sai) vì 3 chưa bao giờ được so
print("Đúng:", ascending_charscan("a 5 b 3"))

Sai : True
Đúng: False


## 13. Mẹo nhận dạng pattern

- "Kiểm tra dãy **tăng / giảm / không đổi**" → chỉ cần so **cặp liền kề**, giữ một biến `prev`.
- "Trong câu có lẫn số và chữ" → `split()` + `isdigit()` hoặc `re.findall(r"\d+")`.
- Số trong chuỗi có thể nhiều chữ số → **ghép** bằng `cur = cur*10 + digit`, đừng xử lý từng ký tự.
- Cần dừng sớm → vòng `for` + `return False` sớm, hoặc `all(...)` (cũng dừng sớm).
- Bài liên quan: 1859 Sorting the Sentence, 1816 Truncate Sentence, 896 Monotonic Array, 2047 Number of Valid Words in a Sentence.

## 14. Biến thể

1. **Không giảm** (cho phép bằng): đổi `x <= prev` thành `x < prev`.
2. **Trả về vị trí vi phạm đầu tiên** (chỉ số token), hoặc `-1`.
3. **Số có thể âm** (`"-3"`): `prev = 0` không còn an toàn → dùng `prev = None` hoặc `float("-inf")`, và nhận diện token số bằng `tok.lstrip("-").isdigit()`.

In [21]:
def ascending_with_negatives(s):
    prev = float("-inf")             # âm vô cực: nhỏ hơn mọi số
    for tok in s.split():
        if tok.lstrip("-").isdigit():
            x = int(tok)
            if x <= prev:
                return False
            prev = x
    return True

print(ascending_with_negatives("t -5 a -2 b 0 c 3"))   # True
print(ascending_with_negatives("t -2 a -5"))            # False

True
False


## 15. Bài tập tự luyện

**Bài 1 — Đoán output:** `"10" < "9"`, `"ab 12".split()`, `"7".isdigit()`, `"a7".isdigit()`, `int("08")`.

**Bài 2 — Tính tay:** lập bảng trace (như mục 5) cho `"a 3 b 10 c 10 d 11"`.

**Bài 3 — TODO:** viết lại `exercise_ascending(s)` với `split` + `prev`.

**Bài 4 — TODO biến thể:** `first_violation(s)` trả chỉ số **token** của số đầu tiên vi phạm, hoặc `-1` nếu tăng.

**Bài 5 — TODO:** `non_decreasing(s)` — cho phép số bằng nhau.

**Bài 6 — Mini challenge:** `ascending_no_split(s)` không dùng `split`, không dùng `re`, không tạo chuỗi mới (không `s + " "`): quét ký tự và xử lý số cuối câu sau vòng lặp.

In [22]:
def exercise_ascending(s):
    # TODO 1: prev = 0
    # TODO 2: duyệt token; nếu là số thì đổi int và so với prev
    # TODO 3: return False sớm nếu vi phạm, cuối cùng return True
    pass

# assert exercise_ascending("hello world 5 x 5") == False

In [23]:
def first_violation(s):
    # TODO: dùng enumerate(s.split())
    pass

# assert first_violation("a 3 b 10 c 10 d 11") == 5
# assert first_violation("1 2 3") == -1

In [24]:
def non_decreasing(s):
    # TODO
    pass

# assert non_decreasing("hello world 5 x 5") == True
# assert non_decreasing("a 6 b 5") == False

In [25]:
def ascending_no_split(s):
    # TODO: cur, prev; sau vòng lặp nhớ xử lý cur còn dở
    pass

# assert ascending_no_split("a 5 b 3") == False
# assert ascending_no_split("1 box 3 x 12") == True

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | chuỗi so sánh từ điển | `isdigit` cần mọi ký tự là số | `True`, `['ab','12']`, `True`, `False`, `8` |
| 2 | 10 rồi 10 là bằng nhau | prev: 0→3→10, gặp 10 | False tại token "10" thứ hai |
| 3 | mục 4 | `if tok[0].isdigit()` | `if x <= prev: return False` |
| 4 | cần chỉ số token | `for i, tok in enumerate(...)` | `return i` thay cho `return False` |
| 5 | chỉ đổi một dấu | `x < prev` mới là vi phạm | giữ nguyên phần còn lại |
| 6 | cách 3 bỏ lính canh | sau `for`, kiểm tra `if cur > 0` | `return cur == 0 or cur > prev` |

## 17. Đáp án tham khảo

**Bài 1:** `True` · `['ab', '12']` · `True` · `False` · `8` (`int` chấp nhận "08", dù đề không có số 0 đầu).

**Bài 2:** a(bỏ) → 3 > 0, prev=3 → b(bỏ) → 10 > 3, prev=10 → c(bỏ) → 10 <= 10 → **False** (token chỉ số 5). Số 11 không được xét.

In [26]:
print("10" < "9", "ab 12".split(), "7".isdigit(), "a7".isdigit(), int("08"))
trace_ascending("a 3 b 10 c 10 d 11")

True ['ab', '12'] True False 8
[ 0] 'a': là từ, bỏ qua
[ 1] '3': x=3 > prev=0 -> prev = 3
[ 2] 'b': là từ, bỏ qua
[ 3] '10': x=10 > prev=3 -> prev = 10
[ 4] 'c': là từ, bỏ qua
[ 5] '10': x=10 <= prev=10 -> return False


False

In [27]:
def exercise_ascending_answer(s):
    prev = 0
    for tok in s.split():
        if tok[0].isdigit():
            x = int(tok)
            if x <= prev:
                return False
            prev = x
    return True

def first_violation_answer(s):
    prev = 0
    for i, tok in enumerate(s.split()):
        if tok[0].isdigit():
            x = int(tok)
            if x <= prev:
                return i
            prev = x
    return -1

def non_decreasing_answer(s):
    prev = 0
    for tok in s.split():
        if tok[0].isdigit():
            x = int(tok)
            if x < prev:
                return False
            prev = x
    return True

def ascending_no_split_answer(s):
    prev = cur = 0
    for ch in s:
        if ch.isdigit():
            cur = cur * 10 + (ord(ch) - ord('0'))
        elif cur > 0:
            if cur <= prev:
                return False
            prev, cur = cur, 0
    return cur == 0 or cur > prev

assert exercise_ascending_answer("hello world 5 x 5") == False
assert first_violation_answer("a 3 b 10 c 10 d 11") == 5
assert first_violation_answer("1 2 3") == -1
assert non_decreasing_answer("hello world 5 x 5") == True
assert non_decreasing_answer("a 6 b 5") == False
assert ascending_no_split_answer("a 5 b 3") == False
assert ascending_no_split_answer("1 box 3 x 12") == True
random.seed(11)
for _ in range(500):
    nums = [random.randint(1, 99) for _ in range(random.randint(2, 6))]
    s = " ".join(f"w {x}" for x in nums)
    assert ascending_no_split_answer(s) == ascending_prev(s) == exercise_ascending_answer(s)
print("Tất cả đáp án bài tập — PASS")

Tất cả đáp án bài tập — PASS


## 18. Tổng kết — template + checklist

**Template kiểm tra dãy tăng nghiêm ngặt trong luồng dữ liệu:**

```python
prev = SMALLER_THAN_ALL        # 0 nếu mọi số dương, float("-inf") nếu có số âm
for item in stream:
    if not is_number(item):
        continue
    x = int(item)
    if x <= prev:              # <= cho "nghiêm ngặt", < cho "không giảm"
        return False
    prev = x
return True
```

**Checklist submit**

- [ ] So sánh **số** (`int`), không so sánh chuỗi.
- [ ] Dấu `<=` cho vi phạm (bằng nhau → False).
- [ ] Số nhiều chữ số được đọc trọn vẹn (`"12"` là 12, không phải 1 và 2).
- [ ] Có cập nhật `prev = x`.
- [ ] Tên method `areNumbersAscending`, trả `bool`.